# 05 - Training

Load model YOLOv8s (như `04_model.ipynb`), cấu hình training theo `configs/training.yaml`, và fine-tune
trên `data/processed/data.yaml` (6.061 ảnh, single-class `Human`).

**Quy trình 2 bước**:
1. **Smoke test** (1-2 epoch) - xác nhận pipeline chạy được, không OOM, logging/checkpoint đúng chỗ, trước khi cam kết chạy full training (có thể mất nhiều giờ trên laptop GPU 6GB VRAM).
2. **Full training** (100 epoch theo config) - cell riêng, chỉ chạy khi đã xác nhận smoke test ổn và ước tính thời gian chấp nhận được.

## 0. Setup

Đọc `configs/training.yaml` + `configs/model.yaml`, build model như notebook 04.

In [1]:
import os
import sys
import shutil
import datetime

import yaml
import torch

sys.path.insert(0, os.path.abspath(".."))
from src.models.model_factory import build_model

with open("../configs/training.yaml", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)
with open("../configs/model.yaml", encoding="utf-8") as f:
    model_cfg = yaml.safe_load(f)

data_yaml_path = os.path.abspath("../data/processed/data.yaml")
if not os.path.isfile(data_yaml_path):
    raise RuntimeError("Chưa có data/processed/data.yaml. Chạy notebook 03_preprocessing.ipynb trước.")

weights_dir = os.path.abspath("../outputs/checkpoints")
logs_dir = os.path.abspath("../outputs/logs")  # chua ca weights/results/tensorboard cua tung run

device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", device, "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("data.yaml:", data_yaml_path)
print("Training config:", train_cfg)

Device: 0 - NVIDIA GeForce RTX 3060 Laptop GPU
data.yaml: D:\Thermal_project\data\processed\data.yaml
Training config: {'seed': 42, 'device': 'cuda', 'model': {'name': 'yolov8'}, 'optimizer': {'name': 'adamw', 'lr': 0.001, 'weight_decay': 0.0005}, 'scheduler': {'name': 'cosine', 'warmup_epochs': 3}, 'training': {'epochs': 100, 'batch_size': 16, 'num_workers': 4, 'image_size': [640, 640], 'amp': True}, 'early_stopping': {'patience': 15, 'monitor': 'val_map50'}, 'checkpoint': {'dir': 'outputs/checkpoints', 'save_best_only': True}, 'logging': {'tensorboard_dir': 'outputs/tensorboard', 'log_dir': 'outputs/logs'}}


In [2]:
model = build_model(model_cfg, weights_dir=weights_dir)
print("Model:", model_cfg["name"], model_cfg["variant"], "| pretrained:", model_cfg["pretrained"])

Model: yolov8 s | pretrained: True


## 1. Ánh xạ `configs/training.yaml` sang tham số `model.train()`

**Lưu ý về early stopping**: `configs/training.yaml` khai báo `monitor: val_map50`, nhưng ultralytics dùng
một chỉ số nội bộ gọi là *fitness* (kết hợp có trọng số giữa mAP50 và mAP50-95, không phải mAP50 đơn thuần)
để quyết định early stop qua tham số `patience` - không có tuỳ chọn giám sát đúng 1 metric string tuỳ ý
trong API chuẩn. Vẫn dùng `patience` từ config, nhưng ghi nhận sai khác này để không hiểu lầm là early stop
chính xác theo mAP50 riêng lẻ.

In [3]:
def build_train_kwargs(epochs, project, name, batch=None):
    return dict(
        data=data_yaml_path,
        epochs=epochs,
        imgsz=train_cfg["training"]["image_size"][0],
        batch=train_cfg["training"]["batch_size"] if batch is None else batch,
        workers=train_cfg["training"]["num_workers"],
        amp=train_cfg["training"]["amp"],
        optimizer=train_cfg["optimizer"]["name"].upper(),
        lr0=train_cfg["optimizer"]["lr"],
        weight_decay=train_cfg["optimizer"]["weight_decay"],
        cos_lr=(train_cfg["scheduler"]["name"] == "cosine"),
        warmup_epochs=train_cfg["scheduler"]["warmup_epochs"],
        patience=train_cfg["early_stopping"]["patience"],
        seed=train_cfg["seed"],
        device=device,
        project=project,
        name=name,
        exist_ok=True,
        verbose=True,
    )

print(build_train_kwargs(epochs=1, project=logs_dir, name="_demo"))

{'data': 'D:\\Thermal_project\\data\\processed\\data.yaml', 'epochs': 1, 'imgsz': 640, 'batch': 16, 'workers': 4, 'amp': True, 'optimizer': 'ADAMW', 'lr0': 0.001, 'weight_decay': 0.0005, 'cos_lr': True, 'warmup_epochs': 3, 'patience': 15, 'seed': 42, 'device': 0, 'project': 'D:\\Thermal_project\\outputs\\logs', 'name': '_demo', 'exist_ok': True, 'verbose': True}


## 2. Smoke test (2 epoch)

Chạy training thật nhưng chỉ 2 epoch trên toàn bộ tập train/val - mục tiêu là xác nhận:
- Data loader đọc đúng ảnh/label từ `data/processed/`
- Checkpoint + log ghi đúng chỗ dưới `outputs/logs/`
- GPU 6GB VRAM chạy được ở `imgsz=1280`

**Về batch size**: `configs/training.yaml` khai báo `batch_size: 16`, nhưng thử nghiệm thực tế cho thấy
GPU 6GB VRAM bị OOM (out of memory) ở `imgsz=1280` với batch này. Quan trọng hơn: một khi CUDA báo OOM,
**toàn bộ CUDA context của tiến trình Python đó bị hỏng** - kể cả gọi `torch.cuda.empty_cache()` sau đó
cũng lỗi theo, nên không thể tự retry với batch nhỏ hơn trong cùng 1 process. Vì vậy dùng
**`batch=-1` (autobatch)** - ultralytics tự dò dung lượng GPU trống và chọn batch an toàn *trước khi*
chạy training thật, tránh hẳn kịch bản OOM giữa chừng.

In [4]:
SMOKE_EPOCHS = 2
smoke_name = "yolov8s_thermal_smoke"

kwargs = build_train_kwargs(epochs=SMOKE_EPOCHS, project=logs_dir, name=smoke_name, batch=-1)
print("Train kwargs (batch=-1 -> autobatch):", kwargs)

smoke_start = datetime.datetime.now()
smoke_results = model.train(**kwargs)
smoke_elapsed = (datetime.datetime.now() - smoke_start).total_seconds()

used_batch = getattr(model.trainer, "batch_size", "không xác định được, xem log autobatch ở trên")
print(f"\nSmoke test xong: batch tự động chọn = {used_batch}, {SMOKE_EPOCHS} epoch trong {smoke_elapsed:.1f}s "
      f"(~{smoke_elapsed / SMOKE_EPOCHS:.1f}s/epoch)")

Train kwargs (batch=-1 -> autobatch): {'data': 'D:\\Thermal_project\\data\\processed\\data.yaml', 'epochs': 2, 'imgsz': 640, 'batch': -1, 'workers': 4, 'amp': True, 'optimizer': 'ADAMW', 'lr0': 0.001, 'weight_decay': 0.0005, 'cos_lr': True, 'warmup_epochs': 3, 'patience': 15, 'seed': 42, 'device': 0, 'project': 'D:\\Thermal_project\\outputs\\logs', 'name': 'yolov8s_thermal_smoke', 'exist_ok': True, 'verbose': True}


Ultralytics 8.4.98  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)


engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\Thermal_project\data\processed\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\Thermal_project\outputs\checkpoints\yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_thermal_smoke, nbs=64, nms=False, opset=None, optimize=False, optimizer=ADAMW, overlap_mask=True, patience=1

Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 76/76 15.0it/s 5.1s

                   all        606        889      0.974      0.959      0.975      0.603


Speed: 0.3ms preprocess, 2.8ms inference, 0.0ms loss, 1.4ms postprocess per image


Results saved to D:\Thermal_project\outputs\logs\yolov8s_thermal_smoke



Smoke test xong: batch tự động chọn = 4, 2 epoch trong 299.3s (~149.7s/epoch)


## 3. Ước tính thời gian full training

In [5]:
per_epoch_sec = smoke_elapsed / SMOKE_EPOCHS
full_epochs = train_cfg["training"]["epochs"]
estimated_total_sec = per_epoch_sec * full_epochs
estimated_hours = estimated_total_sec / 3600

print(f"Batch size se dung cho full training: {used_batch}")
print(f"Uoc tinh: {full_epochs} epoch x ~{per_epoch_sec:.1f}s/epoch = ~{estimated_hours:.1f} gio "
      f"(chua tinh early stopping co the dung som hon neu patience={train_cfg['early_stopping']['patience']} "
      f"epoch khong cai thien)."
)
print("=> Xem lai uoc tinh nay truoc khi chay cell '4. Full training' ben duoi.")

Batch size se dung cho full training: 4
Uoc tinh: 100 epoch x ~149.7s/epoch = ~4.2 gio (chua tinh early stopping co the dung som hon neu patience=15 epoch khong cai thien).
=> Xem lai uoc tinh nay truoc khi chay cell '4. Full training' ben duoi.


## 4. Full training (100 epoch theo config)

**Chỉ chạy cell dưới khi đã xem ước tính thời gian ở mục 3 và chấp nhận thời lượng đó.**
Dùng lại `batch=-1` (autobatch) - để ultralytics tự dò lại dung lượng GPU trống tại thời điểm chạy thật
(độc lập với smoke test, an toàn hơn là tái sử dụng con số batch cũ). Kết quả (weights, log, plot) lưu tại
`outputs/logs/yolov8s_thermal/`; sau khi xong, `best.pt` được copy sang `outputs/checkpoints/best.pt`
để khớp với `configs/inference.yaml` (`checkpoint: outputs/checkpoints/best.pt`).

In [6]:
RUN_FULL_TRAINING = False  # doi thanh True khi da san sang chay that (co the mat nhieu gio)

if not RUN_FULL_TRAINING:
    print("RUN_FULL_TRAINING=False -> bo qua. Doi thanh True o cell tren de chay full training that.")
else:
    full_name = "yolov8s_thermal"
    kwargs = build_train_kwargs(epochs=full_epochs, project=logs_dir, name=full_name, batch=-1)
    full_start = datetime.datetime.now()
    full_results = model.train(**kwargs)
    full_elapsed = (datetime.datetime.now() - full_start).total_seconds()
    print(f"Full training xong trong {full_elapsed / 3600:.2f} gio")

    best_src = os.path.join(logs_dir, full_name, "weights", "best.pt")
    checkpoint_dst = os.path.join(weights_dir, "best.pt")
    if os.path.isfile(best_src):
        shutil.copy2(best_src, checkpoint_dst)
        print("Da copy best checkpoint sang:", checkpoint_dst)
    else:
        print("CANH BAO: khong tim thay", best_src)

RUN_FULL_TRAINING=False -> bo qua. Doi thanh True o cell tren de chay full training that.


## 5. Tóm tắt

In [7]:
print("=" * 60)
print("TÓM TẮT TRAINING")
print("=" * 60)
print(f"Smoke test: {SMOKE_EPOCHS} epoch, batch={used_batch}, {smoke_elapsed:.1f}s "
      f"(~{per_epoch_sec:.1f}s/epoch) - PASS, khong OOM")
print(f"Uoc tinh full training ({full_epochs} epoch): ~{estimated_hours:.1f} gio")
print(f"RUN_FULL_TRAINING = {RUN_FULL_TRAINING}")
print(f"Log/checkpoint tai: {logs_dir}")
print(f"Checkpoint cuoi cung (khi full training xong): {os.path.join(weights_dir, 'best.pt')}")

TÓM TẮT TRAINING
Smoke test: 2 epoch, batch=4, 299.3s (~149.7s/epoch) - PASS, khong OOM
Uoc tinh full training (100 epoch): ~4.2 gio
RUN_FULL_TRAINING = False
Log/checkpoint tai: D:\Thermal_project\outputs\logs
Checkpoint cuoi cung (khi full training xong): D:\Thermal_project\outputs\checkpoints\best.pt
